<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). ترجمته آنا لاريونوفا. تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> الموضوع 6. الانحدار</center>
## <center>انحدارات لاسو وريدج</center>
*يختلف منهج المحاضرات هذا الأسبوع عن مخطط المقالة، لأن الموضوع 4 (النماذج الخطية) ضخم ومهم للغاية، لذلك سنغطي الانحدار هذا الأسبوع.*


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

%config InlineBackend.figure_format = 'retina'
import seaborn as sns

sns.set()  # just to use the seaborn theme


from sklearn.datasets import load_boston
from sklearn.linear_model import Lasso, LassoCV, Ridge, RidgeCV
from sklearn.model_selection import KFold, cross_val_score


**سنعمل مع بيانات أسعار المنازل في بوسطن (مستودع UCI).**
** تنزيل البيانات. **


In [ ]:
boston = load_boston()
X, y = boston["data"], boston["target"]


**دعونا نقرأ وصف البيانات:**


In [ ]:
print(boston.DESCR)

In [ ]:
boston.feature_names


**دعونا نلقي نظرة على أول سجلين.**


In [ ]:
X[:2]


## انحدار لاسو



يقلل انحدار Lasso من متوسط الخطأ التربيعي من خلال تنظيم L1:
$$\Large error(X, y, w) = \frac{1}{2} \sum_{i=1}^\ell {(y_i - w^Tx_i)}^2 + \alpha \sum_{i=1}^d |w_i|$$
حيث $y = w^Tx$ معادلة المستوى التشعبي اعتمادًا على معلمات النموذج $w$، $\ell$ هو عدد الملاحظات في البيانات $X$، $d$ هو عدد الميزات، $y$ القيم المستهدفة، $\alpha$ معامل التنظيم.



**دعونا نلائم انحدار Lasso مع معامل $\alpha$ الصغير (تنظيم ضعيف). المعامل المتعلق بخاصية NOX (تركيز أكاسيد النيتريك) سيكون صفراً. وهذا يعني أن هذه الميزة هي الأقل أهمية للتنبؤ بمتوسط أسعار المنازل في هذه المنطقة.**


In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(X, y)
lasso.coef_

** دعنا ندرب انحدار اللاسو باستخدام $\alpha=10$. جميع المعاملات تساوي الصفر باستثناء ميزات ZN (نسبة الأراضي السكنية المخصصة للقطع التي تزيد مساحتها عن 25000 قدم مربع)، والضريبة (معدل ضريبة الأملاك على القيمة الكاملة)، وB (نسبة السود حسب المدينة) وLSTAT (% من الوضع الأدنى للسكان).**


In [ ]:
lasso = Lasso(alpha=10)
lasso.fit(X, y)
lasso.coef_


**هذا يعني أن انحدار Lasso قد يكون بمثابة طريقة لاختيار الميزة.**


In [ ]:
n_alphas = 200
alphas = np.linspace(0.1, 10, n_alphas)
model = Lasso()

coefs = []
for a in alphas:
    model.set_params(alpha=a)
    model.fit(X, y)
    coefs.append(model.coef_)

plt.rcParams["figure.figsize"] = (12, 8)

ax = plt.gca()
# ax.set_color_cycle(['b', 'r', 'g', 'c', 'k', 'y', 'm'])

ax.plot(alphas, coefs)
ax.set_xscale("log")
ax.set_xlim(ax.get_xlim()[::-1])  # reverse axis
plt.xlabel("alpha")
plt.ylabel("weights")
plt.title("Lasso coefficients as a function of the regularization")
plt.axis("tight")
plt.show();


**الآن لنجد أفضل قيمة لـ $\alpha$ أثناء التحقق المتبادل.**


In [ ]:
lasso_cv = LassoCV(alphas=alphas, cv=3, random_state=17)
lasso_cv.fit(X, y)

In [ ]:
lasso_cv.coef_

In [ ]:
lasso_cv.alpha_


**في Scikit-learn، عادةً ما يتم تكبير المقاييس *لذلك بالنسبة لـ MSE يوجد حل بديل: `neg_mean_squared_error` يتم تصغيرها بدلاً من ذلك. ليست مريحة حقا. **


In [ ]:
cross_val_score(Lasso(lasso_cv.alpha_), X, y, cv=3, scoring="neg_mean_squared_error")

In [ ]:
abs(
    cross_val_score(
        Lasso(lasso_cv.alpha_), X, y, cv=3, scoring="neg_mean_squared_error"
    ).mean()
)

In [ ]:
abs(np.mean(cross_val_score(Lasso(9.95), X, y, cv=3, scoring="neg_mean_squared_error")))


**نقطة أخرى غامضة: يقوم LassoCV بفرز قيم المعلمات بترتيب تنازلي لتسهيل عملية التحسين. قد يبدو أن التحسين $\alpha$ يعمل بشكل غير صحيح.**


In [ ]:
lasso_cv.alphas[:10]

In [ ]:
lasso_cv.alphas_[:10]

In [ ]:
plt.plot(lasso_cv.alphas, lasso_cv.mse_path_.mean(1))  # incorrect
plt.axvline(lasso_cv.alpha_, c="g");

In [ ]:
plt.plot(lasso_cv.alphas_, lasso_cv.mse_path_.mean(1))  # correct
plt.axvline(lasso_cv.alpha_, c="g");


## انحدار ريدج



يقلل انحدار Ridge من متوسط الخطأ التربيعي من خلال تنظيم L2:
$$\Large error(X, y, w) = \frac{1}{2} \sum_{i=1}^\ell {(y_i - w^Tx_i)}^2 + \alpha \sum_{i=1}^d w_i^2$$
حيث $y = w^Tx$ معادلة المستوى التشعبي اعتمادًا على معلمات النموذج $w$، $\ell$ هو عدد الملاحظات في البيانات $X$، $d$ هو عدد الميزات، $y$ القيم المستهدفة، $\alpha$ معامل التنظيم.



هناك فئة خاصة [RidgeCV](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html#sklearn.linear_model.RidgeCV) للتحقق من صحة انحدار Ridge.


In [ ]:
n_alphas = 200
ridge_alphas = np.logspace(-2, 6, n_alphas)

In [ ]:
ridge_cv = RidgeCV(alphas=ridge_alphas, scoring="neg_mean_squared_error", cv=3)
ridge_cv.fit(X, y)

In [ ]:
ridge_cv.alpha_


**في حالة انحدار ريدج، لا ينخفض أي من المعلمات إلى الصفر. يمكن أن تكون قيمة صغيرة ولكن غير صفرية.**


In [ ]:
ridge_cv.coef_

In [ ]:
n_alphas = 200
ridge_alphas = np.logspace(-2, 6, n_alphas)
model = Ridge()

coefs = []
for a in ridge_alphas:
    model.set_params(alpha=a)
    model.fit(X, y)
    coefs.append(model.coef_)

ax = plt.gca()
# ax.set_color_cycle(['b', 'r', 'g', 'c', 'k', 'y', 'm'])

ax.plot(ridge_alphas, coefs)
ax.set_xscale("log")
ax.set_xlim(ax.get_xlim()[::-1])  # reverse axis
plt.xlabel("alpha")
plt.ylabel("weights")
plt.title("Ridge coefficients as a function of the regularization")
plt.axis("tight")
plt.show()


## المراجع
- [النماذج الخطية المعممة](http://scikit-learn.org/stable/modules/linear_model.html) (النماذج الخطية المعممة، GLM) في Scikit-learn
- [الانحدار الخطي](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression)، [Lasso](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html#sklearn.linear_model.Lasso)، [LassoCV](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html#sklearn.linear_model.LassoCV)، [Ridge](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html) و[RidgeCV](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.RidgeCV.html#sklearn.linear_model.RidgeCV) في Scikit-learn